In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
silver_customer=f"{catalog_name}.silver.customer"
silver_taxrate= f"{catalog_name}.silver.taxrate"
staging_customer=f"{catalog_name}.staging.customer_scd2_versions"
gold_customer=f"{catalog_name}.gold.dim_customer"
gold_prospect=f"{catalog_name}.gold.dim_prospect"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_staging=spark.read.table(staging_customer)
df_taxrate=spark.read.table(silver_taxrate)
df_prospect=spark.read.table(gold_prospect)

df_staging.createOrReplaceTempView("df_staging")
df_taxrate.createOrReplaceTempView("df_taxrate")
df_prospect.createOrReplaceTempView("df_prospect")

In [0]:
# df_prospect.printSchema()

In [0]:
# Join National TAX 
df_enriched=spark.sql("""
           select s.*,
            nt.TX_NAME AS nationaltaxratedesc,
            nt.TX_RATE AS nationaltaxrate
           from df_staging s
           left join df_taxrate nt
           on s.C_NAT_TX_ID=nt.TX_ID
                      """)
# df_enriched.limit(10).display()
df_enriched.createOrReplaceTempView("df_enriched")

In [0]:
# Join Local TAX
df_result=spark.sql("""
           select e.*,
            lt.TX_NAME AS localtaxratedesc,
            lt.TX_RATE AS localtaxrate
           from df_enriched e
           left join df_taxrate lt
           on e.C_LCL_TX_ID =lt.TX_ID
                      """)
# print(df_result.count())
df_result.createOrReplaceTempView("df_result")

In [0]:
# df_result.printSchema()

In [0]:
df_final_enriched=spark.sql("""
                select r.*,
                p.agencyid,
                p.creditrating,
                p.networth,
                p.marketingnameplate
                from df_result r
                left join df_prospect p
                on upper(r.C_L_Name)=upper(p.lastname)
                and upper(r.C_F_Name)=upper(p.firstname)
                and upper(r.C_ADLINE1)=upper(p.addressline1)
                and coalesce(upper(r.C_ADLINE2),'')=coalesce(upper(p.addressline2),'')
                and upper(r.C_ZIPCODE)=upper(p.postalcode)
""")

In [0]:
# df_final_enriched.printSchema()

In [0]:
# df_final_enriched.limit(10).display()

In [0]:
df_gold=df_final_enriched.withColumn("sk_customerid",expr("try_cast(concat(date_format(EffectiveDate,'yyyyMMdd'),C_ID) AS bigint)"))\
    .withColumnRenamed("C_ID","customerid")\
    .withColumnRenamed("C_TAX_ID","taxid")\
    .withColumnRenamed("Status","status")\
    .withColumnRenamed("C_L_NAME","lastname")\
    .withColumnRenamed("C_F_NAME","firstname")\
    .withColumnRenamed("C_M_NAME","middleinitial")\
    .withColumn("gender",expr("CASE WHEN UPPER(C_GNDR)='M' THEN 'M' WHEN UPPER(C_GNDR)='F' THEN 'F' ELSE 'U' END"))\
    .withColumn("tier",col("C_TIER").cast("TINYINT"))\
    .withColumnRenamed("C_DOB","dob")\
    .withColumnRenamed("C_ADLINE1","addressline1")\
    .withColumnRenamed("C_ADLINE2","addressline2")\
    .withColumnRenamed("C_ZIPCODE", "postalcode")\
    .withColumnRenamed("C_CITY","city")\
    .withColumnRenamed("C_STATE_PROV","stateprov")\
    .withColumnRenamed("C_CTRY","country")\
    .withColumn("phone1",expr("CASE WHEN C_CTRY_1 IS NOT NULL THEN concat(C_CTRY_1, '-', C_AREA_1, '-', C_LOCAL_1, IF(C_EXT_1 IS NOT NULL, concat(' ext ', C_EXT_1), '')) ELSE NULL END"))\
    .withColumn("phone2",expr("CASE WHEN C_CTRY_2 IS NOT NULL THEN concat(C_CTRY_2, '-', C_AREA_2, '-', C_LOCAL_2, IF(C_EXT_2 IS NOT NULL, concat(' ext ', C_EXT_2), '')) ELSE NULL END"))\
    .withColumn("phone3",expr("CASE WHEN C_CTRY_3 IS NOT NULL THEN concat(C_CTRY_3, '-', C_AREA_3, '-', C_LOCAL_3, IF(C_EXT_3 IS NOT NULL, concat(' ext ', C_EXT_3), '')) ELSE NULL END"))\
    .withColumnRenamed("C_PRIM_EMAIL","primaryemail")\
    .withColumnRenamed("C_ALT_EMAIL","alternateemail")\
    .withColumn("nationaltaxrate",col("nationaltaxrate").cast("DECIMAL(6,4)"))\
    .withColumn("localtaxrate", col("localtaxrate").cast("DECIMAL(6,4)"))\
    .withColumn("agencyid",col("agencyid"))\
    .withColumn("creditrating",col("creditrating"))\
    .withColumn("networth",col("networth"))\
    .withColumn("marketingnameplate",col("marketingnameplate"))\
    .withColumnRenamed("IsCurrent","iscurrent")\
    .withColumn("valid_from",col("EffectiveDate"))\
    .withColumn("valid_to",col("EndDate"))\
    .withColumnRenamed("EffectiveDate", "effectivedate")\
    .withColumnRenamed("EndDate","enddate")\
    .withColumn("system_valid_from", current_timestamp())\
    .withColumn("system_valid_to", expr("try_cast('9999-12-31 23:59:59' as timestamp)"))\
    .withColumn("_load_ts",current_timestamp())

In [0]:
final_columns=["sk_customerid","customerid","taxid","lastname","firstname","middleinitial",
    "gender","tier","dob","addressline1","addressline2","postalcode","city",
    "stateprov","country","phone1","phone2","phone3","primaryemail",
    "alternateemail","nationaltaxratedesc","nationaltaxrate",
    "localtaxratedesc","localtaxrate","agencyid","creditrating","networth",
    "marketingnameplate","iscurrent","valid_from","valid_to","effectivedate",
    "enddate","version_number","record_hash","system_valid_from",
    "system_valid_to","_batch","_run_id","_load_ts"]

In [0]:
df_gold=df_gold.select(*final_columns)
df_gold.createOrReplaceTempView("df_gold")

In [0]:
# df_gold.select("middleinitial").distinct().display()

In [0]:
try:
    #If table is not present then write as it is in case of Batch 1 only 
    if spark.catalog.tableExists(gold_customer):
        print("Merging into gold table started....")
        spark.sql(f"""
                MERGE INTO {gold_customer} t 
                using df_gold s
                on t.customerid=s.customerid and 
                t.effectivedate=s.effectivedate
                WHEN MATCHED THEN
                UPDATE SET *
                WHEN NOT MATCHED THEN
                INSERT *
                """)
        print("Merged Successfully.....")
        operation_type = "MERGE" 
        
    else:
        print("This is Batch 1 so creating new table and inserting data")
        df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_customer)
        print("Data is Write successfully....")
        operation_type = "OVERWRITE"

    run_id=df_gold.select("_run_id").first()[0]
    staging_history = spark.sql(f"DESCRIBE HISTORY {staging_customer}").first()
    source_count = int(staging_history["operationMetrics"].get("numOutputRows", 0))

    gold_history = spark.sql(f"DESCRIBE HISTORY {gold_customer}").first()
    metrics = gold_history["operationMetrics"]

    if operation_type == "MERGE":
        inserted = int(metrics.get("numTargetRowsInserted", 0))
        updated = int(metrics.get("numTargetRowsUpdated", 0))
        deleted = int(metrics.get("numTargetRowsDeleted", 0))
        rows_affected = inserted + updated + deleted
    else: 
        # For Batch 1 OVERWRITE
        rows_affected = int(metrics.get("numOutputRows", 0))
    target_count = spark.read.table(gold_customer).count()

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="dim_customer",
        source_layer="staging",       # Architecture dictates this reads from staging [1]
        target_layer="gold",
        source_count=source_count,
        target_count=target_count
    )

except Exception as e:
    print("Error in writing data to table",e)
    raise e

In [0]:
spark.table(gold_customer).count()

In [0]:
# spark.catalog.tableExists(gold_customer)

In [0]:
# df_gold.select("marketingnameplate").distinct().display()
# df_gold.limit(10).display()